# 01 — Exploración de CHB-MIT

**Etapa 0 / Fase -1: entender el dominio y ver el primer EEG real.**

Este notebook **solo explora**. No contiene lógica reutilizable: toda la lógica
vive en el paquete `neuropilot/` y acá simplemente se consume.

Objetivos:
1. Parsear el `summary` de un paciente y entender su estructura.
2. Cargar un EDF real con MNE (vía `neuropilot.data.loaders`).
3. Ver los canales y una crisis anotada.
4. Escribir el glosario de dominio (Fase -1) con nuestras palabras.

## 0. Setup

Ejecutar desde la raíz del repo con el paquete instalado (`pip install -e ".[dev]"`).
Ajustar `DATA_DIR` a donde tengas descargado CHB-MIT (al menos el paciente `chb01`).

In [ ]:
from pathlib import Path

from neuropilot.data import loaders

# Ajustar a tu ruta local. Estructura esperada: DATA_DIR/chb01/chb01-summary.txt + *.edf
DATA_DIR = Path.home() / "datasets" / "chb-mit"
PATIENT = "chb01"

patient_dir = DATA_DIR / PATIENT
summary_path = patient_dir / f"{PATIENT}-summary.txt"
print("summary:", summary_path, "| existe:", summary_path.exists())

## 1. Parsear el summary

Frecuencia de muestreo, canales y, por archivo, las crisis anotadas.

In [ ]:
summary = loaders.parse_summary(summary_path)

print("Frecuencia de muestreo:", summary.sampling_rate_hz, "Hz")
print("Cantidad de canales:  ", len(summary.channels))
print("Canales:", summary.channels)
print("Cantidad de archivos: ", len(summary.files))

In [ ]:
# Archivos que contienen al menos una crisis
print("Archivos con crisis:", summary.files_with_seizures)

for name in summary.files_with_seizures:
    for i, s in enumerate(summary.seizures_for_file(name), start=1):
        print(f"  {name}  crisis {i}: {s.start_sec:.0f}s -> {s.end_sec:.0f}s  ({s.duration_sec:.0f}s)")

## 2. Cargar un EDF real (el primero con una crisis)

`read_edf` solo lee: no aplica ningún preprocessing (eso es de otra etapa).

In [ ]:
file_name = summary.files_with_seizures[0]
edf_path = patient_dir / file_name

raw = loaders.read_edf(edf_path, preload=True)

print("Archivo:", file_name)
print("Canales:", loaders.get_channels(raw))
print("Frecuencia:", raw.info["sfreq"], "Hz")
print("Duración:  ", raw.times[-1], "s")

## 3. Superponer las crisis sobre la señal

Convertimos las anotaciones del summary a `mne.Annotations` y las adjuntamos al Raw.

In [ ]:
seizures = summary.seizures_for_file(file_name)
annotations = loaders.to_mne_annotations(seizures)
raw.set_annotations(annotations)

print("Anotaciones adjuntadas:", len(annotations))
for onset, dur in zip(annotations.onset, annotations.duration):
    print(f"  crisis en {onset:.0f}s durante {dur:.0f}s")

In [ ]:
# Ventana alrededor de la primera crisis (interactivo con backend Qt/notebook).
start = max(0.0, seizures[0].start_sec - 20)
raw.plot(start=start, duration=80, n_channels=len(raw.ch_names), title=file_name)

## 4. Glosario de dominio (Fase -1)

Completar con tus propias palabras. Si no lo podés explicar, todavía no estás
listo para modelar.

- **EEG:** _..._
- **Canal / montaje:** _..._
- **Crisis focal vs generalizada:** _..._
- **Ictal / interictal / preictal / postictal:** _..._
- **Artefacto:** _..._
- **¿Cómo se etiqueta una crisis y quién lo hace?** _..._

### Próximo paso
Con el dato entendido y cargado, sigue `neuropilot/data/splits.py`: el **split por
paciente** congelado, antes de mirar cualquier métrica.